# Scanner Backtest — Gemelo Digital del Scanner IBKR

Replica minuto a minuto lo que habría visto un scanner en tiempo real,
usando las barras 1-min ya descargadas. Compara la detección del scanner
vs cuándo Finviz lo habría publicado.

**Pregunta central:** ¿Cuántos minutos antes detecta el scanner vs Finviz,
y cuánto move adicional captura esa ventaja?

**Trigger del scanner (configurable):**
```
vol_1min / avg_vol_20d > VOL_THRESHOLD        # spike de volumen
AND price_change_Nmin > PRICE_THRESHOLD       # momentum de precio
AND (opcional) float < FLOAT_MAX              # filtro de float
AND (opcional) range_compression_5d < 12%    # energía acumulada
```


## 1. Configuración del trigger

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Parámetros del trigger (barrer en S6) ─────────────────────────────────────
VOL_THRESHOLD    = 8.0     # vol_1min / avg_vol_20d mínimo
PRICE_THRESHOLD  = 4.0     # % de subida en N minutos para confirmar
PRICE_WINDOW     = 3       # barras de precio para calcular momentum
FLOAT_MAX        = 50e6    # None = sin filtro de float
AVG_VOL_WINDOW   = 20      # días para calcular avg volumen

# ── Paths ─────────────────────────────────────────────────────────────────────
INTRADAY_DB = Path('/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/burst_intraday_cache.db')
DAILY_DB    = Path('/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/burst_daily_cache.db')
FINVIZ_DB   = Path.home() / 'Library/Application Support/finviz-dashboard/finviz_snapshots.db'
NARRATIVE   = Path('/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/reactive_signal/narrative_master.csv')

print("Config OK")
print(f"  VOL_THRESHOLD  : >{VOL_THRESHOLD}x avg vol 20d")
print(f"  PRICE_THRESHOLD: >{PRICE_THRESHOLD}% en {PRICE_WINDOW} barras")
print(f"  FLOAT_MAX      : {'sin filtro' if not FLOAT_MAX else f'<{FLOAT_MAX/1e6:.0f}M'}")


## 2. Cargar datos

In [ ]:
TZ = 'America/New_York'

# ── 2a. Barras intraday 1-min ─────────────────────────────────────────────────
with sqlite3.connect(INTRADAY_DB) as conn:
    bars = pd.read_sql(
        "SELECT symbol AS ticker, date, dt, open, high, low, close, volume FROM bars",
        conn
    )
bars['bar_ts']  = pd.to_datetime(bars['date'] + 'T' + bars['dt']).dt.tz_localize(TZ, ambiguous='NaT', nonexistent='NaT')
bars['mso']     = (bars['bar_ts'].dt.hour - 9) * 60 + bars['bar_ts'].dt.minute - 30
bars            = bars[bars['mso'] >= 0].copy()
bars            = bars.sort_values(['ticker','date','mso']).reset_index(drop=True)
print(f"Intraday bars: {len(bars):,}  |  ticker-days: {bars.groupby(['ticker','date']).ngroups}")

# ── 2b. Avg volumen 20d por ticker+fecha ──────────────────────────────────────
with sqlite3.connect(DAILY_DB) as conn:
    daily = pd.read_sql(
        "SELECT ticker, bar_date, open, high, low, close, volume FROM daily_bars ORDER BY ticker, bar_date",
        conn
    )

# avg_vol_20d = media de los 20 días PREVIOS al burst day (sin incluir el burst day)
daily = daily.sort_values(['ticker','bar_date']).reset_index(drop=True)
daily['avg_vol_20d'] = (
    daily.groupby('ticker')['volume']
    .transform(lambda x: x.shift(1).rolling(AVG_VOL_WINDOW, min_periods=5).mean())
)
# range compression 5d: (high_5d - low_5d) / close
daily['high_5d']          = daily.groupby('ticker')['high'].transform(lambda x: x.shift(1).rolling(5).max())
daily['low_5d']           = daily.groupby('ticker')['low'].transform(lambda x: x.shift(1).rolling(5).min())
daily['range_5d_pct']     = (daily['high_5d'] - daily['low_5d']) / daily['close'] * 100
daily['prev_close']       = daily.groupby('ticker')['close'].shift(1)

daily_idx = daily.set_index(['ticker','bar_date'])
print(f"Daily bars: {len(daily):,}  |  tickers: {daily['ticker'].nunique()}")

# ── 2c. Finviz: primera aparición por (ticker, date) ─────────────────────────
with sqlite3.connect(FINVIZ_DB) as conn:
    snaps = pd.read_sql("""
        SELECT ticker,
               SUBSTR(timestamp,1,10) AS date,
               MIN(timestamp)         AS first_ts,
               CAST(REPLACE(REPLACE(change_pct,'%',''),',','') AS REAL) AS finviz_chg
        FROM snapshots
        WHERE category = 'Top Gainers'
          AND CAST(REPLACE(REPLACE(change_pct,'%',''),',','') AS REAL) >= 15
        GROUP BY ticker, SUBSTR(timestamp,1,10)
    """, conn)

snaps['first_ts_et'] = pd.to_datetime(snaps['first_ts'], utc=True).dt.tz_convert(TZ)
snaps['mso_finviz']  = (snaps['first_ts_et'].dt.hour - 9) * 60 + snaps['first_ts_et'].dt.minute - 30
snaps = snaps[snaps['mso_finviz'] >= 0].copy()
print(f"Finviz burst events: {len(snaps)}  |  MSO mediano: {snaps['mso_finviz'].median():.0f} min")

# ── 2d. Narrative (setup_type, filtros) ──────────────────────────────────────
narrative = pd.read_csv(NARRATIVE) if NARRATIVE.exists() else pd.DataFrame()
print(f"Narrative: {len(narrative)} bursts")


## 3. Float via yahooquery (cache local)

In [ ]:
import json, asyncio, time
from pathlib import Path

FLOAT_CACHE_FILE = Path('/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/reactive_signal/edgar_cache/float_cache.json')

def load_float_cache():
    if FLOAT_CACHE_FILE.exists():
        return json.loads(FLOAT_CACHE_FILE.read_text())
    return {}

def save_float_cache(cache):
    FLOAT_CACHE_FILE.write_text(json.dumps(cache))

def fetch_floats(tickers, cache):
    """Fetch float via yahooquery para tickers no en cache."""
    missing = [t for t in tickers if t not in cache]
    if not missing:
        return cache

    print(f"Fetching float para {len(missing)} tickers via yahooquery...")
    try:
        from yahooquery import Ticker
        batch_size = 20
        for i in range(0, len(missing), batch_size):
            batch = missing[i:i+batch_size]
            try:
                t = Ticker(batch)
                key_stats = t.key_stats
                for sym in batch:
                    stats = key_stats.get(sym, {})
                    if isinstance(stats, dict):
                        cache[sym] = int(stats.get('floatShares', 0) or 0)
                    else:
                        cache[sym] = 0
            except Exception as e:
                for sym in batch:
                    cache[sym] = 0
            time.sleep(0.5)
        save_float_cache(cache)
        print(f"  Floats obtenidos: {sum(1 for v in cache.values() if v > 0)}/{len(cache)}")
    except ImportError:
        print("  yahooquery no disponible — float filter desactivado")
        for t in missing:
            cache[t] = 0
    return cache

# Cargar/fetchear floats para todos los tickers burst
float_cache = load_float_cache()
all_tickers  = bars['ticker'].unique().tolist()
float_cache  = fetch_floats(all_tickers, float_cache)

# Convertir a Series
float_series = pd.Series(float_cache).rename('float_shares')
n_known = (float_series > 0).sum()
print(f"\nFloat conocido: {n_known}/{len(float_series)} tickers")
if n_known > 0:
    print(f"Distribución float (millones):")
    known = float_series[float_series > 0] / 1e6
    print(f"  p25={known.quantile(.25):.1f}M  median={known.median():.1f}M  p75={known.quantile(.75):.1f}M")
    print(f"  <50M: {(known < 50).sum()}  50-200M: {((known>=50)&(known<200)).sum()}  >200M: {(known>=200).sum()}")


## 4. Motor del scanner — replay minuto a minuto

In [ ]:
def replay_scanner(ticker, burst_date,
                   bars_df, daily_idx, float_cache,
                   vol_threshold=VOL_THRESHOLD,
                   price_threshold=PRICE_THRESHOLD,
                   price_window=PRICE_WINDOW,
                   float_max=FLOAT_MAX):
    """
    Replay de barras 1-min para un ticker+fecha.
    Simula el scanner evaluando el trigger en cada barra.

    Retorna dict con:
        triggered     : bool
        mso_scanner   : minuto desde apertura donde dispara
        price_scanner : precio (close) en la barra de trigger
        vol_ratio     : vol_1min / avg_vol_20d en trigger
        price_chg_Nm  : cambio % en price_window barras
        range_5d_pct  : compresión de rango previa
        float_shares  : float del ticker
        n_false_signals: cuántas barras anteriores casi dispararon (vol>4x pero <threshold)
    """
    day = bars_df[(bars_df['ticker'] == ticker) & (bars_df['date'] == burst_date)].copy()
    if day.empty:
        return None

    # Obtener avg_vol_20d del día anterior
    try:
        avg_vol = daily_idx.loc[(ticker, burst_date), 'avg_vol_20d']
        prev_close = daily_idx.loc[(ticker, burst_date), 'prev_close']
        range_5d = daily_idx.loc[(ticker, burst_date), 'range_5d_pct']
    except KeyError:
        # Intentar con el día de trading anterior disponible
        tkr_daily = daily_idx.xs(ticker, level='ticker') if ticker in daily_idx.index.get_level_values('ticker') else None
        if tkr_daily is None or tkr_daily.empty:
            return None
        before = tkr_daily[tkr_daily.index < burst_date]
        if before.empty:
            return None
        avg_vol    = before.iloc[-1]['avg_vol_20d']
        prev_close = before.iloc[-1]['close']
        range_5d   = before.iloc[-1]['range_5d_pct']

    if not avg_vol or pd.isna(avg_vol) or avg_vol <= 0:
        return None

    # Filtro de float
    float_shares = float_cache.get(ticker, 0)
    if float_max and float_shares > 0 and float_shares > float_max:
        return {'triggered': False, 'reason': 'float_too_large',
                'float_shares': float_shares, 'ticker': ticker, 'burst_date': burst_date}

    day = day.sort_values('mso').reset_index(drop=True)
    closes = day['close'].values
    volumes = day['volume'].values
    msos    = day['mso'].values

    triggered      = False
    mso_scanner    = None
    price_scanner  = None
    vol_ratio_trig = None
    price_chg_trig = None
    n_false        = 0

    for i in range(price_window, len(day)):
        vol_ratio = volumes[i] / avg_vol if avg_vol > 0 else 0

        # Near-miss counter (señales débiles)
        if vol_ratio >= vol_threshold * 0.5:
            n_false += 1

        # Momentum de precio: cambio desde hace price_window barras
        price_chg = (closes[i] - closes[i - price_window]) / closes[i - price_window] * 100

        # TRIGGER
        if vol_ratio >= vol_threshold and price_chg >= price_threshold:
            triggered      = True
            mso_scanner    = int(msos[i])
            price_scanner  = float(closes[i])
            vol_ratio_trig = round(vol_ratio, 1)
            price_chg_trig = round(price_chg, 2)
            n_false        = max(0, n_false - 1)  # no contar el trigger como false
            break

    # MFE desde detección scanner hasta EOD
    mfe_scanner = None
    if triggered and mso_scanner is not None:
        subsequent = day[day['mso'] > mso_scanner]
        if not subsequent.empty and price_scanner:
            eod_price   = subsequent.iloc[-1]['close']
            peak_price  = subsequent['high'].max()
            mfe_scanner = round((peak_price - price_scanner) / price_scanner * 100, 2)
            eod_ret_scanner = round((eod_price - price_scanner) / price_scanner * 100, 2)
        else:
            mfe_scanner = 0.0
            eod_ret_scanner = 0.0
    else:
        eod_ret_scanner = None

    return {
        'ticker':          ticker,
        'burst_date':      burst_date,
        'triggered':       triggered,
        'mso_scanner':     mso_scanner,
        'price_scanner':   price_scanner,
        'vol_ratio':       vol_ratio_trig,
        'price_chg_Nm':    price_chg_trig,
        'range_5d_pct':    round(float(range_5d), 2) if range_5d and not pd.isna(range_5d) else None,
        'float_shares':    float_shares,
        'avg_vol_20d':     round(avg_vol),
        'prev_close':      round(float(prev_close), 4) if prev_close and not pd.isna(prev_close) else None,
        'mfe_scanner':     mfe_scanner,
        'eod_ret_scanner': eod_ret_scanner,
        'n_near_misses':   n_false,
    }

print("replay_scanner() definida.")


## 5. Ejecutar replay — scanner vs Finviz

In [ ]:
# Obtener todos los ticker-days con intraday disponible
intraday_pairs = bars.groupby(['ticker','date']).size().reset_index()[['ticker','date']]
intraday_pairs.columns = ['ticker','burst_date']

print(f"Ticker-days con intraday: {len(intraday_pairs)}")
print("Ejecutando replay...")

scan_results = []
for _, row in intraday_pairs.iterrows():
    r = replay_scanner(
        row['ticker'], row['burst_date'],
        bars, daily_idx, float_cache,
    )
    if r:
        scan_results.append(r)

scan = pd.DataFrame(scan_results)
print(f"Resultados: {len(scan)} ticker-days procesados")
print(f"  Scanner disparó   : {scan['triggered'].sum()} ({scan['triggered'].mean():.1%})")
print(f"  No disparó        : {(~scan['triggered']).sum()}")
print()

# Merge con Finviz
scan_vs_fv = scan[scan['triggered']].merge(
    snaps[['ticker','date','mso_finviz','finviz_chg']].rename(columns={'date':'burst_date'}),
    on=['ticker','burst_date'],
    how='left'
)

# Solo los que también aparecieron en Finviz Top Gainers (bursts confirmados)
scan_vs_fv_burst = scan_vs_fv[scan_vs_fv['mso_finviz'].notna()].copy()
# Triggers que NO aparecieron en Finviz = falsos positivos
false_pos = scan_vs_fv[scan_vs_fv['mso_finviz'].isna()]

print(f"Scanner disparó y era burst real : {len(scan_vs_fv_burst)} (True Positives)")
print(f"Scanner disparó y NO era burst   : {len(false_pos)} (False Positives)")
print(f"Precision (TP/total triggered)   : {len(scan_vs_fv_burst)/(len(scan_vs_fv_burst)+len(false_pos))*100:.1f}%")
print()

# Bursts reales que el scanner NO detectó (missed)
burst_pairs_in_scan = set(zip(scan_vs_fv_burst['ticker'], scan_vs_fv_burst['burst_date']))
all_burst_pairs     = set(zip(snaps['ticker'], snaps['date']))
burst_with_intraday = all_burst_pairs & set(zip(intraday_pairs['ticker'], intraday_pairs['burst_date']))
missed = burst_with_intraday - burst_pairs_in_scan
print(f"Bursts reales con intraday data  : {len(burst_with_intraday)}")
print(f"Recall (detectados/total bursts) : {len(scan_vs_fv_burst)/len(burst_with_intraday)*100:.1f}%")
print(f"Missed                           : {len(missed)}")


## 6. Ventaja temporal y de precio vs Finviz

In [ ]:
if len(scan_vs_fv_burst) == 0:
    print("Sin True Positives — ajustar VOL_THRESHOLD o PRICE_THRESHOLD")
else:
    sv = scan_vs_fv_burst.copy()
    sv['time_advantage']  = sv['mso_finviz'] - sv['mso_scanner']  # minutos de ventaja
    sv['price_advantage'] = (sv['finviz_chg'] - sv['price_chg_Nm'])   # % de precio disponible antes de Finviz

    # Move desde scanner hasta EOD
    # Move desde Finviz hasta EOD: necesita precio en mso_finviz
    # Aproximación: finviz_chg es el % desde prev_close, precio_finviz = prev_close*(1+finviz_chg/100)
    sv['price_finviz']    = sv['prev_close'] * (1 + sv['finviz_chg'] / 100)
    # EOD return desde Finviz: necesitamos el close EOD
    eod_prices = bars.groupby(['ticker','date'])['close'].last().rename('eod_close').reset_index()
    eod_prices.columns = ['ticker','burst_date','eod_close']
    sv = sv.merge(eod_prices, on=['ticker','burst_date'], how='left')
    sv['eod_ret_finviz'] = (sv['eod_close'] - sv['price_finviz']) / sv['price_finviz'] * 100

    print("=" * 65)
    print(f"VENTAJA TEMPORAL Y DE PRECIO — {len(sv)} True Positives")
    print("=" * 65)
    print(f"\nVentaja temporal (min antes que Finviz):")
    print(f"  Mediana : {sv['time_advantage'].median():.0f} min")
    print(f"  Media   : {sv['time_advantage'].mean():.1f} min")
    print(f"  p25/p75 : {sv['time_advantage'].quantile(.25):.0f} / {sv['time_advantage'].quantile(.75):.0f} min")
    print(f"  Scanner detectó ANTES que Finviz: {(sv['time_advantage'] > 0).sum()}/{len(sv)} ({(sv['time_advantage'] > 0).mean():.1%})")
    print(f"  Scanner detectó DESPUÉS         : {(sv['time_advantage'] < 0).sum()}/{len(sv)}")
    print()
    print(f"Precio de entrada:")
    print(f"  Entry scanner (median): ${sv['price_scanner'].median():.2f}")
    print(f"  Entry Finviz  (median): ${sv['price_finviz'].median():.2f}")
    print(f"  Diff median           : {((sv['price_finviz']-sv['price_scanner'])/sv['price_scanner']*100).median():+.1f}%")
    print()
    print(f"Return EOD desde entrada:")
    print(f"  Desde scanner : {sv['eod_ret_scanner'].median():+.1f}% (median)  {sv['eod_ret_scanner'].mean():+.1f}% (mean)")
    print(f"  Desde Finviz  : {sv['eod_ret_finviz'].median():+.1f}% (median)  {sv['eod_ret_finviz'].mean():+.1f}% (mean)")
    print(f"  Mejora        : {(sv['eod_ret_scanner']-sv['eod_ret_finviz']).median():+.1f}pp (median)")
    print()
    print(f"MFE desde entrada scanner: {sv['mfe_scanner'].median():+.1f}% (median)")
    print()

    # Falsos positivos
    print(f"Falsos positivos: {len(false_pos)} ({len(false_pos)/(len(sv)+len(false_pos))*100:.1f}% de todos los triggers)")
    print(f"  (tickers que dispararon el trigger pero NO llegaron a Top Gainers Finviz >=15%)")


## 7. Visualización

In [ ]:
if len(scan_vs_fv_burst) < 3:
    print("Insuficientes TPs para visualizar")
else:
    sv = scan_vs_fv_burst.copy()
    sv['time_advantage'] = sv['mso_finviz'] - sv['mso_scanner']
    sv['price_finviz']   = sv['prev_close'] * (1 + sv['finviz_chg'] / 100)
    sv = sv.merge(eod_prices, on=['ticker','burst_date'], how='left')
    sv['eod_ret_finviz']  = (sv['eod_close'] - sv['price_finviz']) / sv['price_finviz'] * 100

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f'Scanner vs Finviz  |  vol>{VOL_THRESHOLD}x  price>{PRICE_THRESHOLD}%/{PRICE_WINDOW}min  '
                 f'|  TP={len(sv)}  FP={len(false_pos)}', fontsize=12)

    # 1. MSO scanner vs MSO Finviz
    ax = axes[0,0]
    ax.scatter(sv['mso_scanner'], sv['mso_finviz'], alpha=0.6, color='steelblue', s=30)
    lim = max(sv['mso_scanner'].max(), sv['mso_finviz'].max()) + 5
    ax.plot([0,lim],[0,lim], 'r--', linewidth=1, label='Scanner = Finviz')
    ax.fill_between([0,lim],[0,lim],[0,0], alpha=0.05, color='green', label='Scanner antes')
    ax.set_xlabel('MSO Scanner (min)')
    ax.set_ylabel('MSO Finviz (min)')
    ax.set_title('Detección: Scanner vs Finviz')
    ax.legend(fontsize=8)

    # 2. Distribución ventaja temporal
    ax = axes[0,1]
    ax.hist(sv['time_advantage'], bins=20, color='steelblue', alpha=0.8, edgecolor='white')
    ax.axvline(0, color='red', linewidth=1.5, linestyle='--', label='Scanner=Finviz')
    ax.axvline(sv['time_advantage'].median(), color='green', linewidth=1.5, label=f"Median={sv['time_advantage'].median():.0f}min")
    ax.set_title('Ventaja temporal (min antes que Finviz)')
    ax.set_xlabel('Minutos de ventaja')
    ax.legend(fontsize=8)

    # 3. EOD return: scanner vs Finviz
    ax = axes[0,2]
    ax.scatter(sv['eod_ret_finviz'], sv['eod_ret_scanner'], alpha=0.6, s=30, color='green')
    lim2 = max(abs(sv['eod_ret_finviz']).max(), abs(sv['eod_ret_scanner']).max()) + 5
    ax.plot([-lim2,lim2],[-lim2,lim2],'r--',linewidth=1)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_xlabel('EOD return desde Finviz (%)')
    ax.set_ylabel('EOD return desde Scanner (%)')
    ax.set_title('Return EOD: scanner vs Finviz entry')

    # 4. MSO scanner distribution
    ax = axes[1,0]
    ax.hist(sv['mso_scanner'], bins=20, color='green', alpha=0.8, edgecolor='white')
    ax.axvline(sv['mso_scanner'].median(), color='red', linewidth=1.5, label=f"Median={sv['mso_scanner'].median():.0f}min")
    ax.set_title('MSO del scanner trigger')
    ax.set_xlabel('Minutos desde apertura')
    ax.legend(fontsize=8)

    # 5. Vol ratio en el trigger
    ax = axes[1,1]
    ax.hist(sv['vol_ratio'].dropna(), bins=20, color='orange', alpha=0.8, edgecolor='white')
    ax.axvline(VOL_THRESHOLD, color='red', linewidth=1.5, linestyle='--', label=f'Threshold={VOL_THRESHOLD}x')
    ax.set_title('Ratio de volumen en trigger')
    ax.set_xlabel('vol_1min / avg_vol_20d')
    ax.legend(fontsize=8)

    # 6. Range compression vs time advantage
    ax = axes[1,2]
    has_range = sv['range_5d_pct'].notna()
    if has_range.sum() > 5:
        ax.scatter(sv.loc[has_range,'range_5d_pct'], sv.loc[has_range,'time_advantage'],
                   alpha=0.6, s=30, color='purple')
        ax.axhline(0, color='red', linewidth=1, linestyle='--')
        ax.set_xlabel('Range 5d % (compresión previa)')
        ax.set_ylabel('Ventaja temporal (min)')
        ax.set_title('Compresión previa vs ventaja detección')
    else:
        ax.text(0.5, 0.5, 'Insuficientes datos\nrange_5d', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Range compression (sin datos)')

    plt.tight_layout()
    fig.savefig('figures/scanner_backtest.png', dpi=130, bbox_inches='tight')
    plt.show()
    print("Guardado en figures/scanner_backtest.png")


## 8. Sweep de parámetros del trigger

In [ ]:
# Barrer vol_threshold y price_threshold para encontrar el balance
# precision vs recall vs ventaja temporal

sweep_rows = []
for vol_t in [4, 6, 8, 10, 15]:
    for price_t in [2, 3, 4, 6, 8]:
        results_sw = []
        for _, row in intraday_pairs.iterrows():
            r = replay_scanner(row['ticker'], row['burst_date'],
                               bars, daily_idx, float_cache,
                               vol_threshold=vol_t, price_threshold=price_t,
                               float_max=None)  # sin float para no sesgar
            if r:
                results_sw.append(r)

        if not results_sw:
            continue
        sw_df = pd.DataFrame(results_sw)
        triggered = sw_df[sw_df['triggered'] == True]

        if triggered.empty:
            continue

        # Merge con Finviz
        t_fv = triggered.merge(
            snaps[['ticker','date','mso_finviz']].rename(columns={'date':'burst_date'}),
            on=['ticker','burst_date'], how='left'
        )
        tp = t_fv[t_fv['mso_finviz'].notna()]
        fp = t_fv[t_fv['mso_finviz'].isna()]
        n_bursts_with_intraday = len(burst_with_intraday)
        recall    = len(tp) / n_bursts_with_intraday if n_bursts_with_intraday > 0 else 0
        precision = len(tp) / len(t_fv) if len(t_fv) > 0 else 0
        time_adv  = (tp['mso_finviz'] - tp['mso_scanner']).median() if len(tp) > 0 else np.nan

        sweep_rows.append({
            'vol_t': vol_t, 'price_t': price_t,
            'n_triggered': len(triggered),
            'TP': len(tp), 'FP': len(fp),
            'precision': round(precision*100,1),
            'recall':    round(recall*100,1),
            'time_adv_median': round(time_adv,1) if not np.isnan(time_adv) else None,
        })

sweep = pd.DataFrame(sweep_rows)
print("SWEEP DE PARÁMETROS")
print("=" * 75)
print(f"{'vol_t':>6} {'price_t':>7} {'triggered':>10} {'TP':>5} {'FP':>5} {'precision':>10} {'recall':>8} {'time_adv(min)':>14}")
print("-" * 75)
for _, r in sweep.sort_values(['precision','recall'], ascending=[False,False]).iterrows():
    ta = f"{r['time_adv_median']:.0f}" if r['time_adv_median'] else 'N/A'
    print(f"{r['vol_t']:>6.0f} {r['price_t']:>7.0f} {r['n_triggered']:>10.0f} {r['TP']:>5.0f} "
          f"{r['FP']:>5.0f} {r['precision']:>9.1f}% {r['recall']:>7.1f}% {ta:>14}")

print()
# Best configs: precision >= 50% and recall >= 30%
best = sweep[(sweep['precision'] >= 50) & (sweep['recall'] >= 30)]
if not best.empty:
    print("CONFIGS CON precision>=50% Y recall>=30%:")
    print(best.sort_values('time_adv_median', ascending=False).to_string(index=False))
else:
    print("Ninguna config alcanza precision>=50% y recall>=30% simultáneamente")
    print("Mejor balance precision/recall:")
    sweep['score'] = sweep['precision'] * 0.5 + sweep['recall'] * 0.5 - sweep['FP'] * 0.2
    print(sweep.sort_values('score', ascending=False).head(5).to_string(index=False))


## 9. Análisis profundo de bursts NO detectados (Missed)

¿Por qué el scanner falla en el 75% de los bursts? Clasificamos cada missed en categorías de causa raíz.

In [ ]:
# ─── Construir dataset de missed con diagnóstico causa raíz ───────────────────
def diagnose_missed(ticker, burst_date, bars_df, daily_idx, float_cache, snaps_df):
    day = bars_df[(bars_df['ticker'] == ticker) & (bars_df['date'] == burst_date)].copy()
    if day.empty:
        return {'ticker': ticker, 'burst_date': burst_date, 'cause': 'NO_BARS', 'details': 'Sin barras intraday',
                'gap_open_pct': None, 'max_vol_ratio': None, 'max_price_chg_3min': None,
                'bars_over_4x': 0, 'bars_over_8x': 0, 'peak_vol_mso': None, 'peak_vol_ratio': None,
                'eod_chg_pct': None, 'float_shares': 0, 'avg_vol_20d': None, 'prev_close': None,
                'n_bars': 0, 'mso_finviz': None}
    try:
        avg_vol    = daily_idx.loc[(ticker, burst_date), 'avg_vol_20d']
        prev_close = daily_idx.loc[(ticker, burst_date), 'prev_close']
    except KeyError:
        tkr_daily = daily_idx.xs(ticker, level='ticker') if ticker in daily_idx.index.get_level_values('ticker') else None
        if tkr_daily is None:
            return {'ticker': ticker, 'burst_date': burst_date, 'cause': 'NO_DAILY', 'details': 'Sin datos diarios',
                    'gap_open_pct': None, 'max_vol_ratio': None, 'max_price_chg_3min': None,
                    'bars_over_4x': 0, 'bars_over_8x': 0, 'peak_vol_mso': None, 'peak_vol_ratio': None,
                    'eod_chg_pct': None, 'float_shares': 0, 'avg_vol_20d': None, 'prev_close': None,
                    'n_bars': len(day), 'mso_finviz': None}
        before = tkr_daily[tkr_daily.index < burst_date]
        if before.empty:
            return {'ticker': ticker, 'burst_date': burst_date, 'cause': 'NO_DAILY', 'details': 'Sin datos diarios previos',
                    'gap_open_pct': None, 'max_vol_ratio': None, 'max_price_chg_3min': None,
                    'bars_over_4x': 0, 'bars_over_8x': 0, 'peak_vol_mso': None, 'peak_vol_ratio': None,
                    'eod_chg_pct': None, 'float_shares': 0, 'avg_vol_20d': None, 'prev_close': None,
                    'n_bars': len(day), 'mso_finviz': None}
        avg_vol    = before.iloc[-1]['avg_vol_20d']
        prev_close = before.iloc[-1]['close']

    if not avg_vol or pd.isna(avg_vol) or avg_vol <= 0:
        return {'ticker': ticker, 'burst_date': burst_date, 'cause': 'NO_AVG_VOL', 'details': f'avg_vol={avg_vol}',
                'gap_open_pct': None, 'max_vol_ratio': None, 'max_price_chg_3min': None,
                'bars_over_4x': 0, 'bars_over_8x': 0, 'peak_vol_mso': None, 'peak_vol_ratio': None,
                'eod_chg_pct': None, 'float_shares': 0, 'avg_vol_20d': avg_vol, 'prev_close': prev_close,
                'n_bars': len(day), 'mso_finviz': None}

    float_shares = float_cache.get(ticker, 0)
    if float_shares > 50e6:
        day2 = day.sort_values('mso').reset_index(drop=True)
        open_price = day2['close'].iloc[0]
        gap_open_pct = (open_price - prev_close) / prev_close * 100 if prev_close else 0
        eod_close = day2['close'].iloc[-1]
        eod_chg = (eod_close - prev_close) / prev_close * 100 if prev_close else None
        fv_row = snaps_df[(snaps_df['ticker'] == ticker) & (snaps_df['date'] == burst_date)]
        mso_fv = fv_row['mso_finviz'].iloc[0] if not fv_row.empty else None
        return {'ticker': ticker, 'burst_date': burst_date, 'cause': 'FLOAT_TOO_LARGE',
                'details': f'float={float_shares/1e6:.0f}M > 50M',
                'gap_open_pct': round(gap_open_pct,1), 'max_vol_ratio': day2['volume'].max()/avg_vol,
                'max_price_chg_3min': 0, 'bars_over_4x': 0, 'bars_over_8x': 0,
                'peak_vol_mso': None, 'peak_vol_ratio': None,
                'eod_chg_pct': round(eod_chg,1) if eod_chg else None,
                'float_shares': float_shares, 'avg_vol_20d': round(avg_vol), 'prev_close': prev_close,
                'n_bars': len(day), 'mso_finviz': mso_fv}

    day = day.sort_values('mso').reset_index(drop=True)
    closes  = day['close'].values
    volumes = day['volume'].values
    msos    = day['mso'].values
    n_bars  = len(day)

    fv_row = snaps_df[(snaps_df['ticker'] == ticker) & (snaps_df['date'] == burst_date)]
    mso_fv = fv_row['mso_finviz'].iloc[0] if not fv_row.empty else None

    vol_ratios  = volumes / avg_vol
    max_vol_ratio = vol_ratios.max()

    open_price = closes[0] if n_bars > 0 else prev_close
    gap_open_pct = (open_price - prev_close) / prev_close * 100 if prev_close else 0

    max_price_chg_3min = 0
    for i in range(3, n_bars):
        if closes[i-3] > 0:
            chg = (closes[i] - closes[i-3]) / closes[i-3] * 100
            if chg > max_price_chg_3min:
                max_price_chg_3min = chg

    eod_close = closes[-1] if n_bars > 0 else None
    eod_chg   = (eod_close - prev_close) / prev_close * 100 if (eod_close and prev_close) else None

    bars_over_4x = int((vol_ratios >= 4.0).sum())
    bars_over_8x = int((vol_ratios >= 8.0).sum())
    peak_vol_mso = int(msos[vol_ratios.argmax()]) if n_bars > 0 else None
    peak_vol_ratio = float(vol_ratios.max())

    if gap_open_pct >= 15:
        cause = 'GAP_OPEN_PREMARKET'
        details = f'gap_open={gap_open_pct:.0f}%'
    elif max_vol_ratio < 4.0:
        cause = 'LOW_VOL_THROUGHOUT'
        details = f'max_vol_ratio={max_vol_ratio:.1f}x'
    elif max_vol_ratio >= 8.0 and max_price_chg_3min < 4.0:
        cause = 'SLOW_GRIND_NO_MOMENTUM'
        details = f'max_vol={max_vol_ratio:.1f}x pero max_price_chg_3min={max_price_chg_3min:.1f}%'
    elif max_vol_ratio < 8.0 and max_price_chg_3min < 4.0:
        cause = 'BOTH_BELOW_THRESHOLD'
        details = f'max_vol={max_vol_ratio:.1f}x, max_price_chg_3min={max_price_chg_3min:.1f}%'
    elif max_vol_ratio >= 8.0 and max_price_chg_3min >= 4.0:
        cause = 'VOL_PRICE_DESYNC'
        details = f'max_vol={max_vol_ratio:.1f}x, max_price_chg={max_price_chg_3min:.1f}% no simultaneos'
    else:
        cause = 'OTHER'
        details = f'max_vol={max_vol_ratio:.1f}x, max_price_chg_3min={max_price_chg_3min:.1f}%'

    return {
        'ticker': ticker, 'burst_date': burst_date, 'cause': cause, 'details': details,
        'gap_open_pct': round(gap_open_pct, 1), 'max_vol_ratio': round(max_vol_ratio, 1),
        'max_price_chg_3min': round(max_price_chg_3min, 1),
        'bars_over_4x': bars_over_4x, 'bars_over_8x': bars_over_8x,
        'peak_vol_mso': peak_vol_mso, 'peak_vol_ratio': round(peak_vol_ratio, 1),
        'eod_chg_pct': round(eod_chg, 1) if eod_chg is not None else None,
        'float_shares': float_shares, 'avg_vol_20d': round(avg_vol),
        'prev_close': round(float(prev_close), 4) if prev_close and not pd.isna(prev_close) else None,
        'n_bars': n_bars, 'mso_finviz': mso_fv,
    }

missed_list = list(missed)
print(f"Analizando {len(missed_list)} bursts no detectados...")
missed_diagnostics = []
for ticker, date in missed_list:
    d = diagnose_missed(ticker, date, bars, daily_idx, float_cache, snaps)
    missed_diagnostics.append(d)

missed_df = pd.DataFrame(missed_diagnostics)
cause_counts = missed_df['cause'].value_counts()
print("\nCAUSAS RAIZ:")
for cause, n in cause_counts.items():
    print(f"  {cause:<30} {n:>4}  ({n/len(missed_df)*100:.1f}%)")

In [ ]:
print("=" * 70)
print("ANALISIS DETALLADO POR CAUSA RAIZ")
print("=" * 70)

colors_cause = {
    'GAP_OPEN_PREMARKET': '#e74c3c', 'LOW_VOL_THROUGHOUT': '#e67e22',
    'SLOW_GRIND_NO_MOMENTUM': '#f39c12', 'FLOAT_TOO_LARGE': '#9b59b6',
    'BOTH_BELOW_THRESHOLD': '#3498db', 'VOL_PRICE_DESYNC': '#1abc9c',
    'NO_DAILY': '#95a5a6', 'NO_AVG_VOL': '#bdc3c7', 'NO_BARS': '#7f8c8d', 'OTHER': '#34495e',
}

for cause in missed_df['cause'].value_counts().index:
    subset = missed_df[missed_df['cause'] == cause]
    n = len(subset)
    print(f"\n{'─'*70}")
    print(f"CAUSA: {cause}  ({n} bursts, {n/len(missed_df)*100:.1f}%)")
    print(f"{'─'*70}")
    has_eod = subset['eod_chg_pct'].notna()

    if cause == 'GAP_OPEN_PREMARKET':
        print(f"  Gap apertura mediano: +{subset['gap_open_pct'].median():.0f}%")
        print(f"  El burst ocurre en pre-market (antes de 9:30 ET). No scannable con RTH 1-min.")
        if has_eod.sum() > 3:
            gap = subset.loc[has_eod, 'gap_open_pct']
            eod = subset.loc[has_eod, 'eod_chg_pct']
            rth_cont = (eod > gap).sum()
            print(f"  Continuacion RTH (EOD > gap): {rth_cont}/{has_eod.sum()} ({rth_cont/has_eod.sum()*100:.0f}%)")
            print(f"  EOD mediano: +{eod.median():.0f}%")
        for _, r in subset.head(4).iterrows():
            print(f"    {r['ticker']} {r['burst_date']}: gap={r['gap_open_pct']:.0f}%  eod={r['eod_chg_pct']}%")

    elif cause == 'LOW_VOL_THROUGHOUT':
        print(f"  Max vol ratio mediano: {subset['max_vol_ratio'].median():.1f}x (threshold 8x)")
        print(f"  EOD change mediano: +{subset['eod_chg_pct'].median():.0f}%")
        print(f"  Subida organica sin spike de volumen. No detectable con vol-scanner.")
        has_gap = (subset['gap_open_pct'] > 5).sum()
        print(f"  De estos, {has_gap} tienen gap_open > 5% (catalyst silencioso)")
        for _, r in subset.head(4).iterrows():
            print(f"    {r['ticker']} {r['burst_date']}: max_vol={r['max_vol_ratio']:.1f}x  gap={r['gap_open_pct']:.0f}%  eod={r['eod_chg_pct']}%")

    elif cause == 'SLOW_GRIND_NO_MOMENTUM':
        print(f"  Tienen vol spike (mediano {subset['max_vol_ratio'].median():.0f}x) PERO subida gradual")
        print(f"  Max price_chg_3min mediano: {subset['max_price_chg_3min'].median():.1f}% (threshold 4%)")
        print(f"  Vol presente, precio lento -> distribucion de acciones, no momentum real")
        for _, r in subset.head(4).iterrows():
            print(f"    {r['ticker']} {r['burst_date']}: vol={r['max_vol_ratio']:.0f}x  3min_chg={r['max_price_chg_3min']:.1f}%  eod={r['eod_chg_pct']}%")

    elif cause == 'FLOAT_TOO_LARGE':
        print(f"  Float mediano: {subset['float_shares'].median()/1e6:.0f}M (filtro <50M activo)")
        for lo, hi in [(50,100),(100,200),(200,500),(500,9999)]:
            n2 = ((subset['float_shares']/1e6 >= lo) & (subset['float_shares']/1e6 < hi)).sum()
            if n2 > 0: print(f"  {lo}-{hi}M: {n2} bursts")

    elif cause == 'BOTH_BELOW_THRESHOLD':
        print(f"  Vol max mediano: {subset['max_vol_ratio'].median():.1f}x  Price chg mediano: {subset['max_price_chg_3min'].median():.1f}%")
        print(f"  Senales debiles. Con vol>4x y price>2% se capturarian algunos.")
        for _, r in subset.head(4).iterrows():
            print(f"    {r['ticker']} {r['burst_date']}: vol={r['max_vol_ratio']:.1f}x  3min_chg={r['max_price_chg_3min']:.1f}%  eod={r['eod_chg_pct']}%")

    elif cause == 'VOL_PRICE_DESYNC':
        print(f"  Vol max: {subset['max_vol_ratio'].median():.0f}x  Price chg max: {subset['max_price_chg_3min'].median():.1f}%")
        print(f"  Vol spike y subida de precio ocurren en momentos distintos.")
        for _, r in subset.head(4).iterrows():
            print(f"    {r['ticker']} {r['burst_date']}: vol={r['max_vol_ratio']:.0f}x  3min_chg={r['max_price_chg_3min']:.1f}%  eod={r['eod_chg_pct']}%")

    else:
        print(f"  Problema de datos o causa desconocida.")

print(f"\n{'='*70}")
print("RESUMEN OPERATIVO")
print(f"{'='*70}")
total = len(missed_df)
n_premarket  = (missed_df['cause'] == 'GAP_OPEN_PREMARKET').sum()
n_low_vol    = (missed_df['cause'] == 'LOW_VOL_THROUGHOUT').sum()
n_slow       = (missed_df['cause'] == 'SLOW_GRIND_NO_MOMENTUM').sum()
n_float      = (missed_df['cause'] == 'FLOAT_TOO_LARGE').sum()
n_both_below = (missed_df['cause'] == 'BOTH_BELOW_THRESHOLD').sum()
n_desync     = (missed_df['cause'] == 'VOL_PRICE_DESYNC').sum()
n_data       = missed_df['cause'].isin(['NO_DAILY','NO_AVG_VOL','NO_BARS']).sum()

structurally_impossible = n_premarket + n_low_vol + n_data
recoverable = n_slow + n_both_below + n_float + n_desync
tp_current = 42
total_bursts_wi = 228

print(f"  Pre-market (no scannable RTH):          {n_premarket:>3} ({n_premarket/total*100:.0f}%)")
print(f"  Sin vol spike (no detectable):          {n_low_vol:>3} ({n_low_vol/total*100:.0f}%)")
print(f"  Vol ok / precio lento (price>2%):       {n_slow:>3} ({n_slow/total*100:.0f}%)")
print(f"  Float > 50M (con 200M: recuperable):    {n_float:>3} ({n_float/total*100:.0f}%)")
print(f"  Ambos bajo umbral (params menores):     {n_both_below:>3} ({n_both_below/total*100:.0f}%)")
print(f"  Vol+precio desincronizados:             {n_desync:>3} ({n_desync/total*100:.0f}%)")
print(f"  Sin datos:                              {n_data:>3} ({n_data/total*100:.0f}%)")
print(f"  ─────────────────────────────────────────────────────")
print(f"  Estructuralmente NO scanneables:        {structurally_impossible:>3} ({structurally_impossible/total*100:.0f}%)")
print(f"  Potencialmente recuperables:            {recoverable:>3} ({recoverable/total*100:.0f}%)")
print(f"  Recall maximo teorico:                  {(tp_current+recoverable)/total_bursts_wi*100:.0f}%")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f'Analisis de {len(missed_df)} Bursts NO Detectados', fontsize=13)

colors_cause = {
    'GAP_OPEN_PREMARKET': '#e74c3c', 'LOW_VOL_THROUGHOUT': '#e67e22',
    'SLOW_GRIND_NO_MOMENTUM': '#f39c12', 'FLOAT_TOO_LARGE': '#9b59b6',
    'BOTH_BELOW_THRESHOLD': '#3498db', 'VOL_PRICE_DESYNC': '#1abc9c',
    'NO_DAILY': '#95a5a6', 'NO_AVG_VOL': '#bdc3c7', 'NO_BARS': '#7f8c8d', 'OTHER': '#34495e',
}

ax = axes[0,0]
cause_cnt = missed_df['cause'].value_counts()
colors_pie = [colors_cause.get(c, '#999') for c in cause_cnt.index]
wedges, texts, autotexts = ax.pie(cause_cnt.values, labels=None, autopct='%1.0f%%',
                                   colors=colors_pie, startangle=90, pctdistance=0.75)
ax.legend([c.replace('_',' ') for c in cause_cnt.index], loc='lower left', fontsize=7, bbox_to_anchor=(-0.2,-0.2))
ax.set_title('Distribucion de causas')

ax = axes[0,1]
gap_data = missed_df[missed_df['cause'] == 'GAP_OPEN_PREMARKET']['gap_open_pct'].dropna()
if len(gap_data) > 0:
    ax.hist(gap_data, bins=15, color='#e74c3c', alpha=0.8, edgecolor='white')
    ax.axvline(gap_data.median(), color='black', linewidth=2, linestyle='--', label=f'Median={gap_data.median():.0f}%')
    ax.set_xlabel('Gap apertura (%)'); ax.set_title(f'GAP_OPEN ({len(gap_data)} bursts)'); ax.legend(fontsize=9)

ax = axes[0,2]
has_vol = missed_df['max_vol_ratio'].notna()
ax.hist(missed_df.loc[has_vol, 'max_vol_ratio'].clip(upper=100), bins=30, color='#3498db', alpha=0.8, edgecolor='white')
ax.axvline(8.0, color='red', linewidth=2, linestyle='--', label='Threshold 8x')
ax.axvline(4.0, color='orange', linewidth=1.5, linestyle='--', label='Threshold 4x')
ax.set_xlabel('Max vol ratio'); ax.set_title('Missed: vol ratio maximo'); ax.legend(fontsize=9)
ax.set_xlim(0,60)

ax = axes[1,0]
has_price = missed_df['max_price_chg_3min'].notna()
ax.hist(missed_df.loc[has_price, 'max_price_chg_3min'].clip(upper=40), bins=25, color='#f39c12', alpha=0.8, edgecolor='white')
ax.axvline(4.0, color='red', linewidth=2, linestyle='--', label='Threshold 4%')
ax.axvline(2.0, color='orange', linewidth=1.5, linestyle='--', label='Threshold 2%')
ax.set_xlabel('Max price change 3min (%)'); ax.set_title('Missed: momentum de precio'); ax.legend(fontsize=9)

ax = axes[1,1]
cause_order = list(missed_df['cause'].value_counts().index[:6])
eod_by_cause = [missed_df.loc[missed_df['cause']==c, 'eod_chg_pct'].dropna().values for c in cause_order]
eod_valid = [(c, v) for c, v in zip(cause_order, eod_by_cause) if len(v) > 0]
if eod_valid:
    bp = ax.boxplot([v for _,v in eod_valid], labels=[c.replace('_','\n')[:18] for c,_ in eod_valid],
                    patch_artist=True, notch=False)
    for patch, (c,_) in zip(bp['boxes'], eod_valid):
        patch.set_facecolor(colors_cause.get(c,'#999')); patch.set_alpha(0.7)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_ylabel('EOD change desde prev_close (%)'); ax.set_title('EOD return por causa raiz')
ax.tick_params(axis='x', labelsize=7)

ax = axes[1,2]
for cause in ['GAP_OPEN_PREMARKET','LOW_VOL_THROUGHOUT','SLOW_GRIND_NO_MOMENTUM','BOTH_BELOW_THRESHOLD','VOL_PRICE_DESYNC','FLOAT_TOO_LARGE']:
    sub = missed_df[missed_df['cause']==cause]
    if len(sub) > 0:
        ax.scatter(sub['max_vol_ratio'].clip(upper=100), sub['max_price_chg_3min'].clip(upper=50),
                   alpha=0.6, s=25, color=colors_cause.get(cause,'#999'), label=cause.replace('_',' ')[:20])
ax.axvline(8.0, color='red', linewidth=1, linestyle='--', alpha=0.5)
ax.axhline(4.0, color='red', linewidth=1, linestyle='--', alpha=0.5)
ax.set_xlabel('Max vol ratio (clip@100)'); ax.set_ylabel('Max price chg 3min % (clip@50)')
ax.set_title('Missed: vol ratio vs momentum precio'); ax.legend(fontsize=6)

plt.tight_layout()
fig.savefig('figures/missed_analysis.png', dpi=130, bbox_inches='tight')
plt.show()
print("Guardado en figures/missed_analysis.png")

In [ ]:
print("RECALL MAXIMO TEORICO DEL SCANNER RTH")
print("=" * 55)
structurally_unscanneable_causes = ['GAP_OPEN_PREMARKET', 'LOW_VOL_THROUGHOUT', 'NO_DAILY', 'NO_AVG_VOL', 'NO_BARS']
n_struct = missed_df['cause'].isin(structurally_unscanneable_causes).sum()
n_recoverable = len(missed_df) - n_struct
tp_current = 42
total_bursts_wi = 228

print(f"Total missed: {len(missed_df)}")
print(f"Estructuralmente no scanneables: {n_struct} ({n_struct/len(missed_df)*100:.0f}%)")
print(f"  -> GAP pre-market: {(missed_df['cause']=='GAP_OPEN_PREMARKET').sum()}")
print(f"  -> Sin vol spike:  {(missed_df['cause']=='LOW_VOL_THROUGHOUT').sum()}")
print(f"  -> Sin datos:      {missed_df['cause'].isin(['NO_DAILY','NO_AVG_VOL','NO_BARS']).sum()}")
print(f"\nPotencialmente recuperables: {n_recoverable} ({n_recoverable/len(missed_df)*100:.0f}%)")
print(f"TP actuales (vol>8x, price>4%): {tp_current}")
print(f"Recall actual: {tp_current/total_bursts_wi*100:.1f}%")
print(f"Recall maximo teorico: {(tp_current+n_recoverable)/total_bursts_wi*100:.0f}%")
print()

# Test con umbrales minimos
low_results = []
for _, row in intraday_pairs.iterrows():
    r = replay_scanner(row['ticker'], row['burst_date'], bars, daily_idx, float_cache,
                       vol_threshold=2.0, price_threshold=1.0, float_max=None)
    if r: low_results.append(r)
lr_df = pd.DataFrame(low_results)
lr_triggered = lr_df[lr_df['triggered']==True]
lr_fv = lr_triggered.merge(snaps[['ticker','date','mso_finviz']].rename(columns={'date':'burst_date'}),
                            on=['ticker','burst_date'], how='left')
lr_tp = lr_fv[lr_fv['mso_finviz'].notna()]
lr_fp = lr_fv[lr_fv['mso_finviz'].isna()]
lr_recall = len(lr_tp)/total_bursts_wi*100
lr_prec   = len(lr_tp)/len(lr_fv)*100 if len(lr_fv)>0 else 0
print(f"Con umbrales minimos (vol>2x, price>1%, sin float filter):")
print(f"  TP={len(lr_tp)}  FP={len(lr_fp)}  Precision={lr_prec:.0f}%  Recall={lr_recall:.1f}%")
print()
print(f"Conclusion: el {n_struct/len(missed_df)*100:.0f}% de los missed son pre-market o sin vol-spike,")
print(f"estructuralmente no detectables con un scanner RTH de barras 1-min.")
print(f"El recall maximo alcanzable con parametros optimos es ~{(tp_current+n_recoverable)/total_bursts_wi*100:.0f}%.")